In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("../data/processed/merged_f1_data.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (6911, 22)


,season,round,race_name,date,driver_id,driver_code,driver_name,constructor_id,constructor,number,grid,position,points,laps,status,fastest_lap_rank,fastest_lap_speed,fastest_lap_time,qualifying_position,Q1,Q2,Q3
0,2010,1,Bahrain Grand Prix,2010-03-14,alonso,ALO,Fernando Alonso,ferrari,Ferrari,8,3,1,25.0,49,Finished,1.0,191.706,NaN,3.0,1:54.612,1:54.172,1:54.608
1,2010,1,Bahrain Grand Prix,2010-03-14,massa,MAS,Felipe Massa,ferrari,Ferrari,7,2,2,18.0,49,Finished,5.0,189.392,NaN,2.0,1:55.313,1:54.331,1:54.242
2,2010,1,Bahrain Grand Prix,2010-03-14,hamilton,HAM,Lewis Hamilton,mclaren,McLaren,2,4,3,15.0,49,Finished,4.0,189.665,NaN,4.0,1:55.341,1:54.707,1:55.217
3,2010,1,Bahrain Grand Prix,2010-03-14,vettel,VET,Sebastian Vettel,red_bull,Red Bull,5,1,4,12.0,49,Finished,12.0,188.627,NaN,1.0,1:55.029,1:53.883,1:54.101
4,2010,1,Bahrain Grand Prix,2010-03-14,rosberg,ROS,Nico Rosberg,mercedes,Mercedes,4,5,5,10.0,49,Finished,13.0,188.599,NaN,5.0,1:55.463,1:54.682,1:55.241


In [3]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

Number of rows: 6911
Number of columns: 22

Column names:
['season', 'round', 'race_name', 'date', 'driver_id', 'driver_code', 'driver_name', 'constructor_id', 'constructor', 'number', 'grid', 'position', 'points', 'laps', 'status', 'fastest_lap_rank', 'fastest_lap_speed', 'fastest_lap_time', 'qualifying_position', 'Q1', 'Q2', 'Q3']


In [4]:
print(df.dtypes)
df.info()

season                   int64
round                    int64
race_name               object
date                    object
driver_id               object
driver_code             object
driver_name             object
constructor_id          object
constructor             object
number                   int64
grid                     int64
position                 int64
points                 float64
laps                     int64
status                  object
fastest_lap_rank       float64
fastest_lap_speed      float64
fastest_lap_time       float64
qualifying_position    float64
Q1                      object
Q2                      object
Q3                      object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6911 entries, 0 to 6910
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   season               6911 non-null   int64  
 1   round                6911 non-null   int64

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
season,6911.0,2017.527854,4.723994,2010.000,2013.00000,2018.000,2022.00000,2025.000
round,6911.0,10.833454,6.050622,1.000,6.00000,11.000,16.00000,24.000
number,6911.0,23.328751,22.113564,1.000,8.00000,17.000,27.00000,99.000
grid,6911.0,10.915208,6.196824,0.000,6.00000,11.000,16.00000,24.000
position,6911.0,11.061351,6.162395,1.000,6.00000,11.000,16.00000,24.000
points,6911.0,4.831790,7.139900,0.000,0.00000,0.000,8.00000,50.000
laps,6911.0,53.424251,17.745247,0.000,51.00000,56.000,66.00000,87.000
fastest_lap_rank,6624.0,10.687651,5.976106,1.000,6.00000,11.000,16.00000,24.000
fastest_lap_speed,1297.0,197.445974,20.535356,89.540,188.47800,196.808,210.15100,247.861
fastest_lap_time,4840.0,205.156601,21.404169,100.615,192.87375,205.377,219.79775,256.100


In [6]:
missing_values = df.isnull().sum()

missing_values = missing_values[
    missing_values > 0
].sort_values(ascending=False)

print("Columns containing missing values:")
display(missing_values)

print("\nTotal missing values:",
      df.isnull().sum().sum())

Columns containing missing values:


fastest_lap_speed      5614
Q3                     3743
fastest_lap_time       2071
Q2                     1905
fastest_lap_rank        287
Q1                      114
qualifying_position      25
dtype: int64


Total missing values: 13759


In [7]:
print(
    "Missing qualifying positions before:",
    df["qualifying_position"].isnull().sum()
)

df["qualifying_position"] = (
    df["qualifying_position"].fillna(df["grid"])
)

print(
    "Missing qualifying positions after:",
    df["qualifying_position"].isnull().sum()
)

Missing qualifying positions before: 25
Missing qualifying positions after: 0


In [8]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

race_driver_duplicates = df.duplicated(
    subset=["season", "round", "driver_id"]
).sum()

print(
    "Duplicate season-round-driver records:",
    race_driver_duplicates
)

Duplicate rows: 0
Duplicate season-round-driver records: 0


In [9]:
numeric_columns = [
    "season",
    "round",
    "number",
    "grid",
    "position",
    "points",
    "laps",
    "fastest_lap_rank",
    "fastest_lap_speed",
    "qualifying_position"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

print("Data types cleaned.")

Data types cleaned.


In [10]:
print("Invalid season values:")
display(
    df.loc[
        ~df["season"].between(2010, 2025),
        ["season"]
    ].drop_duplicates()
)

print("\nInvalid grid values:")
display(
    df.loc[
        df["grid"] < 0,
        ["grid"]
    ].drop_duplicates()
)

print("\nInvalid points values:")
display(
    df.loc[
        df["points"] < 0,
        ["points"]
    ].drop_duplicates()
)

print("\nInvalid laps values:")
display(
    df.loc[
        df["laps"] < 0,
        ["laps"]
    ].drop_duplicates()
)

Invalid season values:


,season



Invalid grid values:


,grid



Invalid points values:


,points



Invalid laps values:


,laps


In [11]:
df = df.sort_values(
    ["season", "round", "date", "driver_id"]
).reset_index(drop=True)

print("Dataset sorted chronologically.")

display(
    df[
        [
            "season",
            "round",
            "race_name",
            "driver_name"
        ]
    ].head(10)
)

Dataset sorted chronologically.


,season,round,race_name,driver_name
0,2010,1,Bahrain Grand Prix,Jaime Alguersuari
1,2010,1,Bahrain Grand Prix,Fernando Alonso
2,2010,1,Bahrain Grand Prix,Rubens Barrichello
3,2010,1,Bahrain Grand Prix,Bruno Senna
4,2010,1,Bahrain Grand Prix,Sébastien Buemi
5,2010,1,Bahrain Grand Prix,Jenson Button
6,2010,1,Bahrain Grand Prix,Karun Chandhok
7,2010,1,Bahrain Grand Prix,Timo Glock
8,2010,1,Bahrain Grand Prix,Lucas di Grassi
9,2010,1,Bahrain Grand Prix,Lewis Hamilton


In [12]:
df["previous_finish"] = (
    df.groupby("driver_id")["position"]
      .shift(1)
)

df[
    [
        "season",
        "round",
        "driver_name",
        "position",
        "previous_finish"
    ]
].head(20)

,season,round,driver_name,position,previous_finish
0,2010,1,Jaime Alguersuari,13,NaN
1,2010,1,Fernando Alonso,1,NaN
2,2010,1,Rubens Barrichello,10,NaN
3,2010,1,Bruno Senna,19,NaN
4,2010,1,Sébastien Buemi,16,NaN
5,2010,1,Jenson Button,7,NaN
6,2010,1,Karun Chandhok,24,NaN
7,2010,1,Timo Glock,20,NaN
8,2010,1,Lucas di Grassi,23,NaN
9,2010,1,Lewis Hamilton,3,NaN


In [13]:
df["avg_finish_all_previous"] = (
    df.groupby("driver_id")["position"]
      .transform(
          lambda x:
          x.shift(1).expanding().mean()
      )
)

df["avg_qualifying_all_previous"] = (
    df.groupby("driver_id")["qualifying_position"]
      .transform(
          lambda x:
          x.shift(1).expanding().mean()
      )
)


In [14]:
df["avg_finish_last_5"] = (
    df.groupby("driver_id")["position"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(5, min_periods=1)
           .mean()
      )
)

df["avg_qualifying_last_5"] = (
    df.groupby("driver_id")["qualifying_position"]
      .transform(
          lambda x:
          x.shift(1)
           .rolling(5, min_periods=1)
           .mean()
      )
)

In [15]:
df["season_points_so_far"] = (
    df.groupby(
        ["season", "driver_id"]
    )["points"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cumsum()
    )
)

In [16]:
df["season_wins_so_far"] = (
    df.groupby(
        ["season", "driver_id"]
    )["position"]
    .transform(
        lambda x:
        (x.shift(1) == 1)
        .fillna(False)
        .cumsum()
    )
)

In [17]:
df["season_podiums_so_far"] = (
    df.groupby(
        ["season", "driver_id"]
    )["position"]
    .transform(
        lambda x:
        (x.shift(1) <= 3)
        .fillna(False)
        .cumsum()
    )
)

In [18]:
df["constructor_points_so_far"] = (
    df.groupby(
        ["season", "constructor"]
    )["points"]
    .transform(
        lambda x:
        x.shift(1)
         .fillna(0)
         .cumsum()
    )
)

In [19]:
df["constructor_wins_so_far"] = (
    df.groupby(
        ["season", "constructor"]
    )["position"]
    .transform(
        lambda x:
        (x.shift(1) == 1)
        .fillna(False)
        .cumsum()
    )
)

In [20]:
df["constructor_podiums_so_far"] = (
    df.groupby(
        ["season", "constructor"]
    )["position"]
    .transform(
        lambda x:
        (x.shift(1) <= 3)
        .fillna(False)
        .cumsum()
    )
)

In [21]:
df["career_races_before"] = (
    df.groupby("driver_id").cumcount()
)

print(
    df[
        [
            "driver_name",
            "season",
            "round",
            "career_races_before"
        ]
    ].head(20)
)

           driver_name  season  round  career_races_before
0    Jaime Alguersuari    2010      1                    0
1      Fernando Alonso    2010      1                    0
2   Rubens Barrichello    2010      1                    0
3          Bruno Senna    2010      1                    0
4      Sébastien Buemi    2010      1                    0
5        Jenson Button    2010      1                    0
6       Karun Chandhok    2010      1                    0
7           Timo Glock    2010      1                    0
8      Lucas di Grassi    2010      1                    0
9       Lewis Hamilton    2010      1                    0
10     Nico Hülkenberg    2010      1                    0
11     Kamui Kobayashi    2010      1                    0
12   Heikki Kovalainen    2010      1                    0
13       Robert Kubica    2010      1                    0
14   Vitantonio Liuzzi    2010      1                    0
15        Felipe Massa    2010      1                   

In [22]:
historical_columns = [
    "previous_finish",
    "avg_finish_all_previous",
    "avg_qualifying_all_previous",
    "avg_finish_last_5",
    "avg_qualifying_last_5"
]

df["previous_finish"] = (
    df["previous_finish"]
    .fillna(df["avg_finish_all_previous"])
)

df["avg_finish_last_5"] = (
    df["avg_finish_last_5"]
    .fillna(df["avg_finish_all_previous"])
)

df["avg_qualifying_last_5"] = (
    df["avg_qualifying_last_5"]
    .fillna(df["avg_qualifying_all_previous"])
)

for column in historical_columns:
    df[column] = df[column].fillna(20)

print(
    df[historical_columns].isnull().sum()
)

previous_finish                0
avg_finish_all_previous        0
avg_qualifying_all_previous    0
avg_finish_last_5              0
avg_qualifying_last_5          0
dtype: int64


In [23]:
df["podium"] = (
    df["position"] <= 3
).astype(int)

print("Podium distribution:")
print(df["podium"].value_counts())

print("\nPodium percentage:")
print(
    df["podium"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Podium distribution:
podium
0    5924
1     987
Name: count, dtype: int64

Podium percentage:
podium
0    85.72
1    14.28
Name: proportion, dtype: float64


In [24]:
feature_columns = [
    "qualifying_position",
    "grid",
    "previous_finish",
    "avg_finish_all_previous",
    "avg_qualifying_all_previous",
    "avg_finish_last_5",
    "avg_qualifying_last_5",
    "season_points_so_far",
    "season_wins_so_far",
    "season_podiums_so_far",
    "constructor_points_so_far",
    "constructor_wins_so_far",
    "constructor_podiums_so_far",
    "career_races_before"
]

X = df[feature_columns].copy()

y_regression = df["position"].copy()

y_classification = df["podium"].copy()

print("X shape:", X.shape)
print("Regression target:", y_regression.shape)
print("Classification target:", y_classification.shape)

X shape: (6911, 14)
Regression target: (6911,)
Classification target: (6911,)


In [25]:
outlier_report = []

for column in feature_columns:

    Q1 = X[column].quantile(0.25)
    Q3 = X[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (X[column] < lower_bound) |
        (X[column] > upper_bound)
    ).sum()

    outlier_report.append({
        "Feature": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outliers
    })

outlier_report = pd.DataFrame(outlier_report)

display(
    outlier_report.sort_values(
        "Outlier Count",
        ascending=False
    )
)

,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
11,constructor_wins_so_far,0.000000,0.000000,0.000000,0.000000,0.000000,1610
9,season_podiums_so_far,0.000000,1.000000,1.000000,-1.500000,2.500000,1258
8,season_wins_so_far,0.000000,0.000000,0.000000,0.000000,0.000000,1120
12,constructor_podiums_so_far,0.000000,3.000000,3.000000,-4.500000,7.500000,1000
7,season_points_so_far,1.000000,60.000000,59.000000,-87.500000,148.500000,701
10,constructor_points_so_far,6.000000,127.000000,121.000000,-175.500000,308.500000,679
13,career_races_before,23.000000,126.000000,103.000000,-131.500000,280.500000,56
3,avg_finish_all_previous,8.497826,13.656483,5.158657,0.759840,21.394469,14
4,avg_qualifying_all_previous,7.831082,14.200000,6.368918,-1.722294,23.753377,7
1,grid,6.000000,16.000000,10.000000,-9.000000,31.000000,0


In [26]:
train_mask = df["season"].between(2010, 2023)

validation_mask = (
    df["season"] == 2024
)

test_mask = (
    df["season"] == 2025
)

In [27]:
X_train = df.loc[
    train_mask,
    feature_columns
].copy()

X_validation = df.loc[
    validation_mask,
    feature_columns
].copy()

X_test = df.loc[
    test_mask,
    feature_columns
].copy()


y_reg_train = df.loc[
    train_mask,
    "position"
].copy()

y_reg_validation = df.loc[
    validation_mask,
    "position"
].copy()

y_reg_test = df.loc[
    test_mask,
    "position"
].copy()


y_clf_train = df.loc[
    train_mask,
    "podium"
].copy()

y_clf_validation = df.loc[
    validation_mask,
    "podium"
].copy()

y_clf_test = df.loc[
    test_mask,
    "podium"
].copy()

In [28]:
print("TRAINING")
print("Seasons: 2010–2023")
print("Rows:", len(X_train))

print("\nVALIDATION")
print("Season: 2024")
print("Rows:", len(X_validation))

print("\nTEST")
print("Season: 2025")
print("Rows:", len(X_test))

TRAINING
Seasons: 2010–2023
Rows: 5953

VALIDATION
Season: 2024
Rows: 479

TEST
Season: 2025
Rows: 479


In [29]:
print("Training missing values:")
print(X_train.isnull().sum().sum())

print("\nValidation missing values:")
print(X_validation.isnull().sum().sum())

print("\nTest missing values:")
print(X_test.isnull().sum().sum())

Training missing values:
0

Validation missing values:
0

Test missing values:
0


In [30]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_validation_scaled = scaler.transform(
    X_validation
)

X_test_scaled = scaler.transform(
    X_test
)

print("Scaling completed.")
print("Scaler fitted only on training data.")

Scaling completed.
Scaler fitted only on training data.


In [31]:
scaled_train = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns
)

print("Training means:")
display(
    scaled_train.mean().round(4)
)

print("\nTraining standard deviations:")
display(
    scaled_train.std().round(4)
)

Training means:


qualifying_position           -0.0
grid                          -0.0
previous_finish                0.0
avg_finish_all_previous       -0.0
avg_qualifying_all_previous    0.0
avg_finish_last_5              0.0
avg_qualifying_last_5          0.0
season_points_so_far          -0.0
season_wins_so_far            -0.0
season_podiums_so_far         -0.0
constructor_points_so_far      0.0
constructor_wins_so_far        0.0
constructor_podiums_so_far    -0.0
career_races_before           -0.0
dtype: float64


Training standard deviations:


qualifying_position            1.0001
grid                           1.0001
previous_finish                1.0001
avg_finish_all_previous        1.0001
avg_qualifying_all_previous    1.0001
avg_finish_last_5              1.0001
avg_qualifying_last_5          1.0001
season_points_so_far           1.0001
season_wins_so_far             1.0001
season_podiums_so_far          1.0001
constructor_points_so_far      1.0001
constructor_wins_so_far        1.0001
constructor_podiums_so_far     1.0001
career_races_before            1.0001
dtype: float64

In [32]:
print("=" * 60)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 60)

print("\nOriginal dataset:", df.shape)

print("\nFeatures:", len(feature_columns))

print("\nTraining:", X_train.shape)
print("Validation:", X_validation.shape)
print("Testing:", X_test.shape)

print("\nRegression target:")
print(y_reg_train.shape)

print("\nClassification target:")
print(y_clf_train.shape)

print("\nMissing values:")
print(
    X_train.isnull().sum().sum(),
    X_validation.isnull().sum().sum(),
    X_test.isnull().sum().sum()
)

FINAL PREPROCESSING SUMMARY

Original dataset: (6911, 35)

Features: 14

Training: (5953, 14)
Validation: (479, 14)
Testing: (479, 14)

Regression target:
(5953,)

Classification target:
(5953,)

Missing values:
0 0 0


In [33]:
OUTPUT_DIR = "../data/processed"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# Full processed master dataset
df.to_csv(
    f"{OUTPUT_DIR}/f1_master_dataset_final.csv",
    index=False
)

# Training / validation / testing
df.loc[train_mask].to_csv(
    f"{OUTPUT_DIR}/train_final.csv",
    index=False
)

df.loc[validation_mask].to_csv(
    f"{OUTPUT_DIR}/validation_final.csv",
    index=False
)

df.loc[test_mask].to_csv(
    f"{OUTPUT_DIR}/test_final.csv",
    index=False
)

# Scaled feature matrices
pd.DataFrame(
    X_train_scaled,
    columns=feature_columns
).to_csv(
    f"{OUTPUT_DIR}/X_train_scaled.csv",
    index=False
)

pd.DataFrame(
    X_validation_scaled,
    columns=feature_columns
).to_csv(
    f"{OUTPUT_DIR}/X_validation_scaled.csv",
    index=False
)

pd.DataFrame(
    X_test_scaled,
    columns=feature_columns
).to_csv(
    f"{OUTPUT_DIR}/X_test_scaled.csv",
    index=False
)

print("All datasets saved successfully!")

All datasets saved successfully!
